# GOLD ATP PLAYERS

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [2]:
try:
    spark = SparkSession.builder.appName("dim_players").getOrCreate()
except Exception as e:
    print(e)

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [ ]:
tb_players = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_players")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

## Players

In [5]:
df = (
        tb_players 
        .withColumn(
            "PLAYER_HAND",
            f.when(f.col("PLAYER_HAND") == 'L', "Left-Handed")
            .when(f.col("PLAYER_HAND") == 'R', "Right-Handed")
            .when(f.col("PLAYER_HAND") == 'A', "Ambidextrous")
            .when(f.col("PLAYER_HAND") == 'U', "Unknown")
            .otherwise("Unknown")
        )
        .withColumn("SK_PLAYER", f.monotonically_increasing_id() + 1)
        .select(
            f.col("SK_PLAYER"),
            f.col("PLAYER_ID"),
            f.col("PLAYER_NAME"),
            f.col("PLAYER_HAND"),
            f.col("PLAYER_HEIGHT"),
            f.col("PLAYER_COUNTRY"),
            f.col("PLAYER_BIRTH_DATE")
        )
    )

## Save dataframe

### Local

In [6]:
df.toPandas().to_csv(
    r"../../../data/gold/dimension/dim_players.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [ ]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_players")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)